# Anomaly-Based Network Intrusion Detection System
## Notebook 02: Data Preprocessing

Covers missing-value imputation, outlier handling, feature scaling,
categorical encoding, and train/validation/test splitting.

## 1. Imports

In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from src.data.loader import DataLoader
from src.data.preprocessor import Preprocessor

print('Ready')

Ready


## 2. Load Raw Data

In [2]:
loader = DataLoader(dataset='unsw_nb15')
df = loader.load_raw_data()
print(f'Loaded {df.shape[0]:,} rows × {df.shape[1]} columns')

Loaded 175,341 rows × 45 columns


## 3. Missing Value Imputation

In [3]:
prep = Preprocessor()
df_clean = prep.handle_missing_values(df)
print(f'Missing after imputation: {df_clean.isnull().sum().sum()}')

Missing after imputation: 0


## 4. Outlier Handling

In [4]:
num_cols = df_clean.select_dtypes(include=['int64','float64']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['label']]

df_clipped = prep.clip_outliers(df_clean, num_cols, lower_q=0.01, upper_q=0.99)
print('Outlier clipping applied (1st–99th percentile)')

Outlier clipping applied (1st–99th percentile)


## 5. Categorical Encoding

In [5]:
cat_cols = df_clipped.select_dtypes(include=['object']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['label', 'attack_cat']]
print(f'Encoding {len(cat_cols)} categorical columns: {cat_cols}')

df_encoded, encoders = prep.encode_categoricals(df_clipped, cat_cols)
print('Encoding complete')

Encoding 3 categorical columns: ['proto', 'service', 'state']
Encoding complete


## 6. Train / Validation / Test Split

In [6]:
TARGET = 'label'
FEATURES = [c for c in df_encoded.columns if c not in ['label', 'attack_cat']]

X = df_encoded[FEATURES]
y = df_encoded[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train : {X_train.shape[0]:,}')
print(f'Val   : {X_val.shape[0]:,}')
print(f'Test  : {X_test.shape[0]:,}')

Train : 122,738
Val   : 26,301
Test  : 26,302


## 7. Feature Scaling

In [7]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('Scaling applied (StandardScaler fit on training set only)')

Scaling applied (StandardScaler fit on training set only)


## 8. Save Processed Data

In [8]:
import joblib

out_dir = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(out_dir, exist_ok=True)

pd.DataFrame(X_train_sc, columns=FEATURES).to_csv(f'{out_dir}/X_train.csv', index=False)
pd.DataFrame(X_val_sc,   columns=FEATURES).to_csv(f'{out_dir}/X_val.csv',   index=False)
pd.DataFrame(X_test_sc,  columns=FEATURES).to_csv(f'{out_dir}/X_test.csv',  index=False)
y_train.to_csv(f'{out_dir}/y_train.csv', index=False)
y_val.to_csv(  f'{out_dir}/y_val.csv',   index=False)
y_test.to_csv( f'{out_dir}/y_test.csv',  index=False)

joblib.dump(scaler,   f'{out_dir}/scaler.pkl')
joblib.dump(encoders, f'{out_dir}/encoders.pkl')

print(f'Saved processed splits to {out_dir}')

Saved processed splits to D:\Github\Anomaly-Based-Neetwork-Intrusion-Detection-System\data\processed
